In [ ]:
import os
import pandas as pd
import numpy as np
import re
import requests
import json

### Loading and Cleaning politician names

In [ ]:
dfPoliticians = pd.read_csv(os.path.join('output', 'politicians.csv'))

dfPoliticians['parsed name'] = dfPoliticians['politician'].str.lower().str.strip()

# remove party labels (e.g. (Rep))
partyLabels = {'(rep)', '(dem)', '(grn)', '(nopty)', '(lbt)', '(ind)', '(w/i)', '(incumbent)'}
for label in partyLabels:
    dfPoliticians['parsed name'] = dfPoliticians['parsed name'].apply(lambda x: x.replace(label, '')).str.strip()

# remove jr. sr. suffixes
dfPoliticians['parsed name'] = dfPoliticians['parsed name'].apply(lambda x: x.replace('jr.', '')).str.strip()
dfPoliticians['parsed name'] = dfPoliticians['parsed name'].apply(lambda x: x.replace('sr.', '')).str.strip()

# make sure name is in first name last name formatting: if comma exist, swap the name on both sides of the comma
for ix in range(0, len(dfPoliticians)):
    name = dfPoliticians.at[ix, 'parsed name']
    if ',' in name:
        _nameList = name.split(',')
        dfPoliticians.at[ix, 'parsed name'] = _nameList[1].strip() + ' ' +  _nameList[0].strip()
        

# remove informal name if first name is present e.g. James "Jim" Jones --> James Jones, but
# "Jim" Jones is untouched, except with " " removed
for ix in range(0, len(dfPoliticians)):
    name = dfPoliticians.at[ix, 'parsed name']
    
    if len(name.split(' ')) > 2:
        if '"' in name:
            name = re.sub('"[^"]+"', '', name)  # remove informal name
            name = re.sub(' +', ' ', name) # remove extra whitespaces
            
            dfPoliticians.at[ix, 'parsed name'] = name
    else: # remove quotation marks
        dfPoliticians.at[ix, 'parsed name'] = name.replace('"', '')

### Getting ethnic data using NamePrism API
Six ethnicity/races are considered in our ethnicity classifier: White, Black, API (Asian and Pacific Islander), AIAN (American Indian and Alaska Native), 2PRACE (more than 2 race) and Hispanic.

In [ ]:
myToken = 'd0424fdc58aecda1'

nameprism = 'http://www.name-prism.com/api_token/eth/json/'

df = pd.DataFrame(columns=['name', 'parsed name', 'White', 'Black', 'Hispanic', 'API', 'AIAN', '2PRACE', 'status', 'notJSON'])

In [ ]:
for ix in range(0, len(dfPoliticians)):
    name = dfPoliticians.at[ix, 'politician']
    parsedName = dfPoliticians.at[ix, 'parsed name']
    url = nameprism + myToken + '/' + requests.utils.quote(parsedName)
    
    r = requests.get(url)
    
    if r.status_code != 200:
        payload['name'] = name
        payload['parsed name'] = parsedName
        payload['status'] = 'not OK'
    else:
        try:
            payload = json.loads(r.text)
            payload['notJSON'] = 0
            payload['status'] = 'OK'
            payload['name'] = name
            payload['parsed name'] = parsedName
            _df = pd.DataFrame.from_dict(payload, orient='index').transpose()
            df = df.append(_df, ignore_index=True, sort=False)
        except:
            payload['notJSON'] = 1

**Check**

In [ ]:
df['notJSON'].sum() 

In [ ]:
len(df[df['status']=='not OK'])

In [ ]:
df.drop(['notJSON', 'status', 'parsed name'], axis=1, inplace=True)

In [ ]:
df.to_csv(os.path.join('output', 'politicians-ethnic.csv'), index=False)

### Parsing the ethnic data into race indicators
* ignore 2PRACE (drop it)
* Race indicators are
    * NP-black
    * NP-white
    * NP-hispanic
    * NP-asian
* Also generate 2nd race indicator, with a value for probabilistic difference between 1st and 2nd race    

In [ ]:
df = pd.read_csv(os.path.join('output', 'politicians-ethnic.csv'))

politicians = df['name'].tolist()
a = df.copy()

a.set_index('name', inplace=True)
a = a.transpose()

In [ ]:
dfRace = pd.DataFrame(columns=['name', 'race 1', 'race 2', 'race difference'])

In [ ]:
for ix, politician in enumerate(politicians):
    _d = a[politician].nlargest(2)
    firstRace, secondRace = _d.index
    difference = _d[0] - _d[1]
    
    dfRace.at[ix, 'name'] = politician
    dfRace.at[ix, 'race 1'] = firstRace
    dfRace.at[ix, 'race 2'] = secondRace
    dfRace.at[ix, 'race difference'] = difference

In [ ]:
dfRace['np1-white'] = np.where(dfRace['race 1']=='White', 1, 0)
dfRace['np2-white'] = np.where(dfRace['race 2']=='White', 1, 0)

dfRace['np1-black'] = np.where(dfRace['race 1']=='Black', 1, 0)
dfRace['np2-black'] = np.where(dfRace['race 2']=='Black', 1, 0)

dfRace['np1-hispanic'] = np.where(dfRace['race 1']=='Hispanic', 1, 0)
dfRace['np2-hispanic'] = np.where(dfRace['race 2']=='Hispanic', 1, 0)

dfRace['np1-hispanic'] = np.where(dfRace['race 1']=='Hispanic', 1, 0)
dfRace['np2-hispanic'] = np.where(dfRace['race 2']=='Hispanic', 1, 0)

dfRace['np1-others'] = np.where(dfRace['np1-white'] + dfRace['np1-black'] + dfRace['np1-hispanic'] == 0, 1, 0)
dfRace['np2-others'] = np.where(dfRace['np2-white'] + dfRace['np2-black'] + dfRace['np2-hispanic'] == 0, 1, 0)

In [ ]:
dfRace.drop(['race 1', 'race 2'], axis=1, inplace=True)

In [ ]:
dfRace.to_csv(os.path.join('output', 'politicians-ethnic-indicators.csv'), index=False)

### Merge with politician data

In [ ]:
dfRace.head()

In [ ]:
dfPoliticians.head()

In [ ]:
dfPoliticians = dfPoliticians.merge(dfRace, right_on='name', left_on='politician')

In [ ]:
dfPoliticians[(dfPoliticians['black']=='yes') & (dfPoliticians['np1-black']==0)]

In [ ]:
# how many I identifed as black, but NP identified as non-black, and are from the main parties
dfPoliticians[(dfPoliticians['black']=='yes') & (dfPoliticians['np1-black']==0) 
              & ((dfPoliticians['party']=='dem')|(dfPoliticians['party']=='rep'))]

In [ ]:
dfPoliticians[(dfPoliticians['black']=='no') & (dfPoliticians['np1-black']==1)]

In [ ]:
dfPoliticians.to_csv(os.path.join('output', 'politicians-with-raceindicators.csv'), index=False)